google collab depedencies

In [1]:
# !pip -q install bertopic
# !pip -q install sastrawi
# !pip -q install gensim

In [2]:
# !git clone -q -b gavriel-thesis https://github.com/ranslemus/topic_modeling_KBMI4.git
# %cd topic_modeling_KBMI4

In [3]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import plotly.express as px

from transformers import AutoTokenizer, AutoModel
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from tqdm.auto import tqdm
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from hdbscan.validity import validity_index

# for linux
from cuml.manifold import UMAP
from cuml.cluster import HDBSCAN

# for windows
# from umap import UMAP
# from hdbscan import HDBSCAN

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device :", device)

if device.type == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))

Device : cuda
GPU : NVIDIA GeForce RTX 3060


In [357]:
df = pd.read_csv("preprocessed_data_2025.csv")
# df = df[df['year'] == 2025]
df = df[df['bank'] == "LIVIN_MANDIRI_REVIEWS"]
# df = df[df['bank'] == "BRIMO_REVIEWS"]
# df = df[df['bank'] == "WONDR_BNI_REVIEWS"]
# df = df[df['bank'] == "BCAMOBILE_REVIEWS"]
df.head()

,reviewId,bank,score,year,text
26099,1f37febf-2a53-41ef-9fa4-eafa90f47b4f,LIVIN_MANDIRI_REVIEWS,1,2025,pemeliharaan setiap hari di jam 11 malam coba ...
26100,f23a9394-1c1f-4153-8925-3a7c7b12a157,LIVIN_MANDIRI_REVIEWS,2,2025,sering pemeliharaan sistem sangat menggangu ap...
26101,510323be-8097-45d6-ac07-4de32d161b71,LIVIN_MANDIRI_REVIEWS,1,2025,kagak jelas tiap malam pemeliharaan
26102,4bd0b173-87cc-41f9-b887-8b46d97e4d91,LIVIN_MANDIRI_REVIEWS,1,2025,kecewa karena pembayaran saya di tunda dan tid...
26103,1fc2108c-6029-4b9f-9b1c-2bad55c182ed,LIVIN_MANDIRI_REVIEWS,2,2025,tolong kirim secepatnya no kartu saya belum ad...


In [358]:
print(f"total dokumen sebelum filter: {len(df):,}")

total dokumen sebelum filter: 14,408


In [359]:
# df["word_count"] = df["text"].astype(str).str.split().apply(len)
# df = df[df["word_count"] >= 5].reset_index(drop=True)
# print(f"Total documents setelah filter: {len(df):,}")

In [360]:
documents = df["text"].astype(str).tolist()

print(f"Total documents : {len(documents):,}")

Total documents : 14,408


# SIMCSE IndoBERT

In [361]:
from sentence_transformers import SentenceTransformer

# Gunakan SimCSE untuk menekan anisotropy dan merapatkan klaster
embedding_model = SentenceTransformer("LazarusNLP/simcse-indobert-base", device=device)

embeddings = embedding_model.encode(
    documents,
    batch_size=128,             # GPU T4/V100 Colab sanggup menangani batch 128 untuk 60k data
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True   # Wajib: Memaksa vektor berukuran L2=1 agar Cosine Distance presisi
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/113 [00:00<?, ?it/s]

# BERTopic

In [362]:
# embeddings = np.load("indobert_embeddings.npy")

print("Embedding Shape :", embeddings.shape)

Embedding Shape : (14408, 768)


stop words

In [363]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /home/pc/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [364]:
from nltk.corpus import stopwords as nltk_stopwords
from bertopic.vectorizers import ClassTfidfTransformer

sastrawi_stopwords = StopWordRemoverFactory().get_stop_words()

# Pure stopwords gabungan (NLTK + Sastrawi)
pure_stopwords = list(set(nltk_stopwords.words('indonesian')).union(set(sastrawi_stopwords)))

topic_stopwords = list(set(
    pure_stopwords + [
        "brimo",
        "livin",
        "mandiri",
        "bca",
        "bni",
        "wondr",
    ]
))


vectorizer_model = CountVectorizer(
    ngram_range=(1, 2),
    stop_words=topic_stopwords,
    token_pattern=r"(?u)\b[^\d\W]{2,}\b",
    min_df=5, # Untuk 60k data, min_df=5 efektif membuang kata typo langka
    # max_df=0.80
)

# Strict c-TF-IDF Transformer untuk memotong frequent words antar-klaster
ctfidf_model = ClassTfidfTransformer(
    reduce_frequent_words=True,
    bm25_weighting=True,
)

baseline UMAP for testing purpose

In [365]:
umap_model = UMAP(
    n_neighbors=15,
    n_components=10,
    metric="cosine",
    min_dist=0.0,
    random_state=42
)

baseline HDBSCAN

In [366]:
hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    min_samples=5,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

K-Means

In [367]:
# k_values = [4, 6, 8, 10, 12]
# results = []

# for bank_name in df["bank"].unique():
#     bank_df = df[df["bank"] == bank_name].copy()

#     bank_documents = bank_df["text"].astype(str).tolist()
#     bank_embeddings = embedding_model.encode(
#         bank_documents,
#         batch_size=128,
#         convert_to_numpy=True,
#         normalize_embeddings=True,
#         show_progress_bar=False
#     )

#     for k in k_values:
#         if len(bank_embeddings) <= k:
#             continue

#         kmeans = KMeans(
#             n_clusters=k,
#             init="k-means++",
#             n_init=10,
#             max_iter=300,
#             random_state=42
#         )

#         labels = kmeans.fit_predict(bank_embeddings)

#         silhouette = silhouette_score(
#             bank_embeddings,
#             labels,
#             metric="cosine"
#         )

#         results.append({
#             "bank": bank_name,
#             "k": k,
#             "inertia": kmeans.inertia_,
#             "silhouette": silhouette
#         })

# results_df = pd.DataFrame(results)

# display(
#     results_df.sort_values(
#         by=["bank", "silhouette"],
#         ascending=[True, False]
#     )
# )

In [368]:
from sklearn.cluster import KMeans

kmeans_model = KMeans(
    n_clusters=10,
    init="k-means++",
    n_init=10,
    max_iter=300,
    random_state=42,
)

In [369]:
topic_model = BERTopic(
    embedding_model=None,
    calculate_probabilities=False,
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    hdbscan_model=kmeans_model,
    ctfidf_model=ctfidf_model,
    verbose=True
)

In [370]:
print(vectorizer_model.min_df)
print(vectorizer_model.max_df)

5
1.0


In [371]:
topics, probabilities = topic_model.fit_transform(
    documents,
    embeddings
)

2026-09-17 21:40:34,255 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-09-17 21:40:34,685 - BERTopic - Dimensionality - Completed ✓
2026-09-17 21:40:34,687 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-09-17 21:40:34,951 - BERTopic - Cluster - Completed ✓
2026-09-17 21:40:34,955 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-09-17 21:40:35,313 - BERTopic - Representation - Completed ✓


In [372]:
# print(vectorizer_model.get_params())

outliers removal

In [373]:
# reduced_topics = topic_model.reduce_outliers(
#         documents,
#         topics,
#         strategy="embeddings",
#         threshold=0.70,
#         embeddings=embeddings
#     )

In [374]:
# topic_model.update_topics(
#     documents,
#     topics=reduced_topics
# )

# Evaluation for Topic Quality

Basic Statistics

In [375]:
topic_info = topic_model.get_topic_info()

# topic_info.head(10)

In [376]:
num_topics = len(
    topic_info[topic_info["Topic"] != -1]
)

outlier_count = (np.array(topics) == -1).sum()

outlier_percentage = (
    outlier_count / len(topics)
) * 100

print(f"Topics              : {num_topics}")
print(f"Outliers            : {outlier_count:,}")
print(f"Outlier Percentage  : {outlier_percentage:.2f}%")

Topics              : 10
Outliers            : 0
Outlier Percentage  : 0.00%


Topic Size

In [377]:
topic_info[["Topic","Count"]]

,Topic,Count
0,0,1828
1,1,1827
2,2,1823
3,3,1751
4,4,1706
5,5,1405
6,6,1404
7,7,998
8,8,860
9,9,806


In [378]:
top_10_topics = topic_model.get_topic_info()
top_10_topics = top_10_topics[top_10_topics.Topic != -1].nlargest(10, "Count")

for _, row in top_10_topics.iterrows():
    topic_id = row['Topic']
    doc_count = row['Count']

    print("=" * 80)
    print(f"TOPIC {topic_id} | JUMLAH DOKUMEN: {doc_count}")
    print("=" * 80)

    # Menampilkan word-score pair bawaan BERTopic (c-TF-IDF scores)
    words_with_scores = topic_model.get_topic(topic_id)
    for word, score in words_with_scores:
        print(f"  - {word:<20} : {score:.4f}")
    print()

TOPIC 0 | JUMLAH DOKUMEN: 1828
  - android              : 0.4270
  - dibuka               : 0.4255
  - hp                   : 0.4182
  - buka                 : 0.3933
  - login                : 0.3836
  - mendukung            : 0.3793
  - operasi              : 0.3766
  - sistem operasi       : 0.3571
  - aplikasinya          : 0.3491
  - resmi                : 0.3368

TOPIC 1 | JUMLAH DOKUMEN: 1827
  - update               : 0.5089
  - lemot                : 0.4285
  - loading              : 0.3899
  - bagus                : 0.3726
  - jaringan             : 0.3259
  - aplikasinya          : 0.3259
  - sinyal               : 0.3217
  - apk                  : 0.3200
  - update update        : 0.3172
  - banget               : 0.3129

TOPIC 2 | JUMLAH DOKUMEN: 1823
  - kartu                : 0.3894
  - rekening             : 0.3869
  - atm                  : 0.3806
  - debit                : 0.3358
  - kartu debit          : 0.3216
  - kartu atm            : 0.3199
  - no               

Representative Reviews

In [379]:
topic_info = topic_model.get_topic_info()

top_10_topics = (
    topic_info[
        topic_info["Topic"] != -1
    ]
    .nlargest(10, "Count")["Topic"]
    .tolist()
)

for topic_id in top_10_topics:
    cluster_size = topic_info.loc[
        topic_info["Topic"] == topic_id,
        "Count"
    ].iloc[0]

    keywords = ", ".join(
        word
        for word, _ in topic_model.get_topic(topic_id)[:10]
    )

    representative_docs = (
        topic_model
        .get_representative_docs(topic_id)
    )

    print(f"\nTOPIC {topic_id}")
    print(f"Cluster size: {cluster_size}")
    print(f"Keywords: {keywords}")
    print("Representative reviews:")

    for index, review in enumerate(
        representative_docs[:3],
        start=1
    ):
        print(f"\n  Review {index}:")
        print(f"  {review}")

    print("\n" + "-" * 80)


TOPIC 0
Cluster size: 1828
Keywords: android, dibuka, hp, buka, login, mendukung, operasi, sistem operasi, aplikasinya, resmi
Representative reviews:

  Review 1:
  saya pakai samsung a14 5g android 13 ketika mau masuk sistem operasi tidak mendukung terus harus pakai hp seperti apa

  Review 2:
  kenapa lagi deh nih aplikasi enggak bisa dibuka katanya sistem operasi handphone tidak mendukung padahal kemarin bisa ini malah enggak bisa hp saya model terbaru 2029 nih wooy android 50

  Review 3:
  tolong di perbaiki kenapa di hp xiaomi kami tidak bisa login dan itu banyak pengguna hp xiaomi juga mengalami hal yang serupa padahal kami beli handphone itu garansi resmi bukan inter alasan sistem operasi handphone anda tidak mendukung untuk penggunaan livin demi keamanan gunakan handphone dengan sistem operasi yang resmi device mi10 5g versi android 13 tolong di perbaiki kami sebagai nasabah mandiri merasa di rugikan karena tidak bisa menggunakan aplikasi livin di hp xiaomi

-----------------

silhoutte score

In [380]:
from sklearn.metrics import silhouette_score

mask = np.array(topics) != -1

silhouette = silhouette_score(
    topic_model.umap_model.embedding_[mask],
    np.array(topics)[mask],
    metric="cosine"
)

print(f"Silhouette Score : {silhouette:.4f}")

Silhouette Score : 0.3778


topic diversity

In [381]:
from itertools import chain

top_n = 10
topic_words = []

for topic in topic_info["Topic"]:
    if topic == -1:
        continue

    words = [
        word
        for word, score in topic_model.get_topic(topic)[:top_n]
    ]

    topic_words.append(words)

flat_words = list(chain.from_iterable(topic_words))

unique_words = len(set(flat_words))
total_words = len(flat_words)

topic_diversity = unique_words / total_words

print(f"Topic Diversity : {topic_diversity:.4f}")

Topic Diversity : 0.8400


NPMI

In [382]:
analyzer = topic_model.vectorizer_model.build_analyzer()

In [383]:
tokenized_docs = [
    analyzer(doc)
    for doc in documents
]

In [384]:
from gensim.corpora import Dictionary

dictionary = Dictionary(tokenized_docs)

top_n = 10
topic_words = []

for topic in topic_info["Topic"]:

    if topic == -1:
        continue

    words = [
        word
        for word, score in topic_model.get_topic(topic)[:top_n]
        if word in dictionary.token2id
    ]

    if len(words) >= 2:
        topic_words.append(words)

print(f"Valid Topics for NPMI: {len(topic_words)}")

Valid Topics for NPMI: 10


In [385]:
from gensim.models.coherencemodel import CoherenceModel

coherence_model = CoherenceModel(
    topics=topic_words,
    texts=tokenized_docs,
    dictionary=dictionary,
    coherence="c_npmi"
)

npmi = coherence_model.get_coherence()
print(f"NPMI : {npmi:.4f}")

NPMI : 0.1113


In [386]:
per_topic = np.array(
    coherence_model.get_coherence_per_topic()
)

overall = coherence_model.get_coherence()

print("Gensim overall :", overall)
print("Mean per-topic :", per_topic.mean())
print("Difference     :", overall - per_topic.mean())

Gensim overall : 0.11125143947737663
Mean per-topic : 0.11125143947737663
Difference     : 0.0


Intertopic Distance Map

In [387]:
fig_intertopic = topic_model.visualize_topics()
fig_intertopic.show()

Topic-Word Scores (barchart c-TF-IDF)

In [388]:
fig_barchart = topic_model.visualize_barchart(
    top_n_topics=len(topic_info[topic_info.Topic != -1]),  # semua topik, atau ganti angka spesifik misal 12
    n_words=10
)
fig_barchart.show()

# Evaluation for Clustering Quality


DBCV -> Only if using HDBSCAN method

In [389]:
# mask = np.array(topics) != -1
# X = topic_model.umap_model.embedding_[mask].astype(np.float64)
# labels = np.array(topics)[mask]

# dbcv_score = validity_index(X, labels)
# print(f"DBCV : {dbcv_score:.4f}")

In [390]:
# import pandas as pd
# from scipy.stats import chi2_contingency

# df["topic"] = topics

# # 1. Baseline: proporsi tiap bank di keseluruhan korpus
# baseline = df["bank"].value_counts(normalize=True) * 100
# print("Proporsi bank di keseluruhan korpus (baseline):")
# print(baseline.round(2))
# print()

# # 2. Proporsi tiap bank DI DALAM tiap topik
# crosstab = pd.crosstab(df["topic"], df["bank"], normalize="index") * 100
# crosstab = crosstab.round(2)

# # 3. Hitung "lift" = proporsi di topik / proporsi baseline
# #    >1 artinya over-represented di topik itu, <1 artinya under-represented
# lift = crosstab.copy()
# for bank in baseline.index:
#     lift[bank] = crosstab[bank] / baseline[bank]

# # 4. Tandai topik yang "njomplang" (deviasi lift > 1.5x atau < 0.5x dari baseline)
# def flag_imbalance(row):
#     return any(row > 1.5) or any(row < 0.5)

# lift["is_imbalanced"] = lift[baseline.index].apply(flag_imbalance, axis=1)

# # gabung count per topik biar gampang liat mana yang topik "besar" (bukan cuma noise kecil)
# topic_sizes = df[df["topic"] != -1]["topic"].value_counts()
# lift["topic_size"] = lift.index.map(topic_sizes)

# result = lift[lift.index != -1].sort_values("is_imbalanced", ascending=False)
# print(result[list(baseline.index) + ["is_imbalanced", "topic_size"]])

checking outliers

In [391]:
# import itertools

# param_grid = {
#     "min_cluster_size": [30, 50, 75],
#     "min_samples": [10, 15, 20],
#     "cluster_selection_method": ["eom", "leaf"],
# }

# results = []
# combos = list(itertools.product(*param_grid.values()))
# print(f"Total kombinasi: {len(combos)}")

# for mcs, ms, method in combos:
#     hdbscan_test = HDBSCAN(
#         min_cluster_size=mcs, min_samples=ms, metric="euclidean",
#         cluster_selection_method=method, prediction_data=True,
#     )
#     tm = BERTopic(
#         embedding_model=None, calculate_probabilities=False,
#         vectorizer_model=vectorizer_model, umap_model=umap_model,
#         hdbscan_model=hdbscan_test, verbose=False,
#     )
#     tpcs, _ = tm.fit_transform(documents, embeddings)

#     ti = tm.get_topic_info()
#     n_topics = len(ti) - 1
#     outlier_pct = (np.array(tpcs) == -1).sum() / len(tpcs) * 100
#     max_share = ti[ti.Topic != -1]["Count"].max() / len(tpcs) * 100 if n_topics > 0 else 0
#     mask = np.array(tpcs) != -1
#     sil = silhouette_score(tm.umap_model.embedding_[mask], np.array(tpcs)[mask]) if len(set(np.array(tpcs)[mask])) > 1 else float("nan")

#     row = {"min_cluster_size": mcs, "min_samples": ms, "method": method,
#            "topics": n_topics, "outlier_%": round(outlier_pct, 2),
#            "max_topic_share_%": round(max_share, 2), "silhouette": round(sil, 4)}
#     results.append(row)
#     print(row)

# results_df = pd.DataFrame(results).sort_values("outlier_%")
# results_df

# eksperimenting to reduce outlier

In [392]:
# # ============================================================
# # TWO-STAGE CLUSTERING ON HDBSCAN OUTLIERS
# # ============================================================

# import numpy as np
# from bertopic import BERTopic
# from hdbscan import HDBSCAN

# # ------------------------------------------------------------
# # 1. Get original outliers
# # ------------------------------------------------------------

# outlier_mask = np.array(topics) == -1

# outlier_documents = [
#     doc for doc, is_outlier in zip(documents, outlier_mask)
#     if is_outlier
# ]

# outlier_embeddings = embeddings[outlier_mask]

# print("Original documents :", len(documents))
# print("Stage 1 outliers   :", len(outlier_documents))


# # ------------------------------------------------------------
# # 2. Create SECOND HDBSCAN
# # ------------------------------------------------------------

# hdbscan_stage2 = HDBSCAN(
#     min_cluster_size=20,
#     min_samples=5,
#     metric="euclidean",
#     cluster_selection_method="eom",
#     prediction_data=True
# )


# # ------------------------------------------------------------
# # 3. Create SECOND BERTopic
# # ------------------------------------------------------------

# topic_model_stage2 = BERTopic(
#     embedding_model=None,
#     calculate_probabilities=False,
#     vectorizer_model=vectorizer_model,
#     umap_model=umap_model,
#     hdbscan_model=hdbscan_stage2,
#     ctfidf_model=ctfidf_model,
#     verbose=True
# )


# # ------------------------------------------------------------
# # 4. Cluster ONLY the original outliers
# # ------------------------------------------------------------

# stage2_topics, _ = topic_model_stage2.fit_transform(
#     outlier_documents,
#     outlier_embeddings
# )


# # ------------------------------------------------------------
# # 5. Stage 2 results
# # ------------------------------------------------------------

# stage2_topics = np.array(stage2_topics)

# stage2_outliers = np.sum(stage2_topics == -1)
# stage2_clustered = len(stage2_topics) - stage2_outliers

# print("\n" + "=" * 60)
# print("STAGE 2 RESULTS")
# print("=" * 60)

# print(f"Input to Stage 2 : {len(outlier_documents):,}")
# print(f"New clusters     : {len(set(stage2_topics)) - (1 if -1 in stage2_topics else 0):,}")
# print(f"Clustered        : {stage2_clustered:,}")
# print(f"Remaining        : {stage2_outliers:,}")
# print(f"Remaining %      : {stage2_outliers / len(stage2_topics) * 100:.2f}%")

In [393]:
# # ============================================================
# # TOP 10 STAGE-2 TOPICS
# # ============================================================

# topic_info_stage2 = topic_model_stage2.get_topic_info()

# display(
#     topic_info_stage2[
#         topic_info_stage2["Topic"] != -1
#     ].head(10)
# )

In [394]:
# # ============================================================
# # REPRESENTATIVE DOCUMENTS - STAGE 2
# # ============================================================

# for topic in topic_info_stage2[
#     topic_info_stage2["Topic"] != -1
# ]["Topic"].head(10):

#     print("=" * 100)
#     print(f"TOPIC {topic}")
#     print("=" * 100)

#     print("KEYWORDS:")
#     print(
#         [
#             word
#             for word, score
#             in topic_model_stage2.get_topic(topic)[:10]
#         ]
#     )

#     docs_topic = topic_model_stage2.get_representative_docs(topic)

#     print("\nREPRESENTATIVE DOCUMENTS:")

#     for i, doc in enumerate(docs_topic[:3], 1):
#         print(f"{i}. {doc}")

#     print()

In [395]:
# # ============================================================
# # COMBINE STAGE 1 + STAGE 2
# # ============================================================

# import numpy as np

# stage1_topics = np.array(topics)
# stage2_topics = np.array(stage2_topics)

# # Start from original Stage 1 labels
# combined_topics = stage1_topics.copy()

# # Get highest topic ID from Stage 1
# valid_stage1_topics = stage1_topics[stage1_topics != -1]

# next_topic_id = (
#     valid_stage1_topics.max() + 1
#     if len(valid_stage1_topics) > 0
#     else 0
# )

# # ------------------------------------------------------------
# # Remap Stage 2 topics so they don't overlap with Stage 1
# # ------------------------------------------------------------

# stage2_valid_topics = sorted(
#     set(stage2_topics) - {-1}
# )

# stage2_mapping = {
#     old_topic: next_topic_id + i
#     for i, old_topic in enumerate(stage2_valid_topics)
# }

# # ------------------------------------------------------------
# # Replace Stage 1 outliers with Stage 2 cluster labels
# # ------------------------------------------------------------

# stage2_positions = np.where(stage1_topics == -1)[0]

# for position, stage2_topic in zip(
#     stage2_positions,
#     stage2_topics
# ):
#     if stage2_topic != -1:
#         combined_topics[position] = stage2_mapping[stage2_topic]

# # ------------------------------------------------------------
# # RESULT
# # ------------------------------------------------------------

# original_outliers = np.sum(stage1_topics == -1)
# remaining_outliers = np.sum(combined_topics == -1)
# final_clustered = len(combined_topics) - remaining_outliers

# print("=" * 60)
# print("COMBINED TWO-STAGE CLUSTERING")
# print("=" * 60)

# print(f"Original documents : {len(combined_topics):,}")
# print(f"Stage 1 outliers   : {original_outliers:,}")
# print(f"Stage 2 recovered  : {original_outliers - remaining_outliers:,}")
# print(f"Final clustered    : {final_clustered:,}")
# print(f"Final outliers     : {remaining_outliers:,}")
# print(
#     f"Final outlier %    : "
#     f"{remaining_outliers / len(combined_topics) * 100:.2f}%"
# )

# print(f"\nStage 1 topics     : {len(set(stage1_topics) - {-1})}")
# print(f"Stage 2 new topics : {len(stage2_valid_topics)}")
# print(
#     f"Combined topics    : "
#     f"{len(set(combined_topics) - {-1})}"
# )

In [396]:
# print("Combined topics :", len(set(combined_topics) - {-1}))
# print("Combined outliers:", sum(t == -1 for t in combined_topics))

In [397]:
# eval_topics = np.array(combined_topics)

# # ============================================================
# # 1. OUTLIER
# # ============================================================

# outlier_count = np.sum(eval_topics == -1)
# outlier_pct = outlier_count / len(eval_topics) * 100

# print("=" * 60)
# print("TWO-STAGE CLUSTERING EVALUATION")
# print("=" * 60)

# print(f"Topics   : {len(set(eval_topics) - {-1})}")
# print(f"Outliers : {outlier_count:,}")
# print(f"Outlier %: {outlier_pct:.2f}%")


# # ============================================================
# # 2. SILHOUETTE
# # ============================================================

# mask = eval_topics != -1

# silhouette = silhouette_score(
#     embeddings[mask],
#     eval_topics[mask]
# )

# print(f"Silhouette : {silhouette:.4f}")


# # ============================================================
# # 3. TOPIC WORDS
# # ============================================================

# topic_words_combined = []

# # Stage 1 topics
# stage1_valid = sorted(set(stage1_topics) - {-1})

# for topic in stage1_valid:
#     words_scores = topic_model.get_topic(topic)

#     if words_scores:
#         topic_words_combined.append([
#             word
#             for word, score in words_scores[:10]
#         ])


# # Stage 2 topics
# for old_topic in stage2_valid_topics:
#     words_scores = topic_model_stage2.get_topic(old_topic)

#     if words_scores:
#         topic_words_combined.append([
#             word
#             for word, score in words_scores[:10]
#         ])


# # ============================================================
# # 4. TOPIC DIVERSITY
# # ============================================================

# unique_words = len(
#     set(
#         word
#         for words in topic_words_combined
#         for word in words
#     )
# )

# total_words = len(topic_words_combined) * 10

# topic_diversity = (
#     unique_words / total_words
# )

# print(f"Topic Diversity : {topic_diversity:.4f}")


# # ============================================================
# # 5. NPMI
# # ============================================================

# coherence_model = CoherenceModel(
#     topics=topic_words_combined,
#     texts=tokenized_docs,
#     dictionary=dictionary,
#     coherence="c_npmi"
# )

# npmi = coherence_model.get_coherence()

# print(f"NPMI : {npmi:.4f}")

In [398]:
# import time 

# min_cluster_sizes = [20, 30, 40, 50, 60, 75, 100]
# min_samples_list = [1, 3, 5, 10]

# results = []


# # ------------------------------------------------------------
# # HELPER: TOPIC DIVERSITY
# # ------------------------------------------------------------

# def calculate_topic_diversity(model, top_n=10):

#     topic_words = []

#     valid_topics = [
#         topic for topic in model.get_topic_info()["Topic"]
#         if topic != -1
#     ]

#     for topic in valid_topics:

#         words_scores = model.get_topic(topic)

#         if words_scores:
#             words = [
#                 word
#                 for word, score in words_scores[:top_n]
#             ]

#             topic_words.append(words)

#     if not topic_words:
#         return np.nan

#     unique_words = len(
#         set(
#             word
#             for words in topic_words
#             for word in words
#         )
#     )

#     total_words = len(topic_words) * top_n

#     return unique_words / total_words


# # ------------------------------------------------------------
# # HELPER: NPMI
# # ------------------------------------------------------------

# def calculate_npmi(model, tokenized_docs, dictionary, top_n=10):

#     topic_words = []

#     valid_topics = [
#         topic for topic in model.get_topic_info()["Topic"]
#         if topic != -1
#     ]

#     for topic in valid_topics:

#         words_scores = model.get_topic(topic)

#         if words_scores:
#             words = [
#                 word
#                 for word, score in words_scores[:top_n]
#             ]

#             topic_words.append(words)

#     if not topic_words:
#         return np.nan

#     coherence_model = CoherenceModel(
#         topics=topic_words,
#         texts=tokenized_docs,
#         dictionary=dictionary,
#         coherence="c_npmi"
#     )

#     return coherence_model.get_coherence()


# # ------------------------------------------------------------
# # EXPERIMENT LOOP
# # ------------------------------------------------------------

# for min_cluster_size in min_cluster_sizes:

#     for min_samples in min_samples_list:

#         print("\n" + "=" * 80)
#         print(
#             f"TESTING: "
#             f"min_cluster_size={min_cluster_size}, "
#             f"min_samples={min_samples}"
#         )
#         print("=" * 80)

#         start_time = time.time()

#         try:

#             # ------------------------------------------------
#             # HDBSCAN
#             # ------------------------------------------------

#             hdbscan_model_test = HDBSCAN(
#                 min_cluster_size=min_cluster_size,
#                 min_samples=min_samples,
#                 metric="euclidean",
#                 cluster_selection_method="eom",
#                 prediction_data=True
#             )

#             # ------------------------------------------------
#             # BERTopic
#             # ------------------------------------------------

#             topic_model_test = BERTopic(
#                 embedding_model=None,
#                 calculate_probabilities=False,
#                 vectorizer_model=vectorizer_model,
#                 umap_model=umap_model,
#                 hdbscan_model=hdbscan_model_test,
#                 ctfidf_model=ctfidf_model,
#                 verbose=False
#             )

#             # ------------------------------------------------
#             # FIT
#             # ------------------------------------------------

#             test_topics, _ = topic_model_test.fit_transform(
#                 documents,
#                 embeddings
#             )

#             test_topics = np.array(test_topics)

#             # ------------------------------------------------
#             # BASIC STATISTICS
#             # ------------------------------------------------

#             total_docs = len(test_topics)

#             outlier_count = np.sum(test_topics == -1)

#             outlier_pct = (
#                 outlier_count / total_docs * 100
#             )

#             num_topics = len(
#                 set(test_topics) - {-1}
#             )

#             # ------------------------------------------------
#             # SILHOUETTE
#             # ------------------------------------------------

#             valid_mask = test_topics != -1

#             unique_valid_topics = len(
#                 set(test_topics[valid_mask])
#             )

#             if (
#                 unique_valid_topics >= 2
#                 and np.sum(valid_mask) > unique_valid_topics
#             ):

#                 silhouette = silhouette_score(
#                     embeddings[valid_mask],
#                     test_topics[valid_mask]
#                 )

#             else:

#                 silhouette = np.nan

#             # ------------------------------------------------
#             # TOPIC DIVERSITY
#             # ------------------------------------------------

#             topic_diversity = calculate_topic_diversity(
#                 topic_model_test
#             )

#             # ------------------------------------------------
#             # NPMI
#             # ------------------------------------------------

#             npmi = calculate_npmi(
#                 topic_model_test,
#                 tokenized_docs,
#                 dictionary
#             )

#             # ------------------------------------------------
#             # TIME
#             # ------------------------------------------------

#             elapsed = time.time() - start_time

#             # ------------------------------------------------
#             # SAVE
#             # ------------------------------------------------

#             results.append({
#                 "min_cluster_size": min_cluster_size,
#                 "min_samples": min_samples,
#                 "num_topics": num_topics,
#                 "outliers": outlier_count,
#                 "outlier_pct": outlier_pct,
#                 "silhouette": silhouette,
#                 "npmi": npmi,
#                 "topic_diversity": topic_diversity,
#                 "runtime_sec": elapsed
#             })

#             print(
#                 f"Topics       : {num_topics}"
#             )

#             print(
#                 f"Outliers     : "
#                 f"{outlier_count:,} "
#                 f"({outlier_pct:.2f}%)"
#             )

#             print(
#                 f"Silhouette   : "
#                 f"{silhouette:.4f}"
#             )

#             print(
#                 f"NPMI         : "
#                 f"{npmi:.4f}"
#             )

#             print(
#                 f"Topic Div.   : "
#                 f"{topic_diversity:.4f}"
#             )

#             print(
#                 f"Runtime      : "
#                 f"{elapsed:.1f}s"
#             )

#         except Exception as e:

#             print(
#                 f"ERROR: {str(e)}"
#             )

#             results.append({
#                 "min_cluster_size": min_cluster_size,
#                 "min_samples": min_samples,
#                 "num_topics": np.nan,
#                 "outliers": np.nan,
#                 "outlier_pct": np.nan,
#                 "silhouette": np.nan,
#                 "npmi": np.nan,
#                 "topic_diversity": np.nan,
#                 "runtime_sec": np.nan
#             })


# # ------------------------------------------------------------
# # FINAL GIANT TABLE
# # ------------------------------------------------------------

# experiment_results = pd.DataFrame(results)

# experiment_results = experiment_results.sort_values(
#     by="silhouette",
#     ascending=False
# ).reset_index(drop=True)


# print("\n" + "=" * 100)
# print("HDBSCAN PARAMETER SWEEP RESULTS")
# print("=" * 100)

# display(experiment_results)

In [399]:
# from bertopic import BERTopic
# from sklearn.feature_extraction.text import CountVectorizer
# import numpy as np

# topics_array = np.array(topics)
# refined_topics = topics_array.copy()

# skipped_topics = []

# for t in set(topics_array):
#     if t == -1:
#         continue
#     idx = np.where(topics_array == t)[0]
#     if len(idx) < 20:
#         continue

#     sub_embeddings = embeddings[idx]
#     sub_documents = [documents[i] for i in idx]

#     sub_vectorizer = CountVectorizer(
#         ngram_range=(1, 2),
#         stop_words=pure_stopwords,   # tetap pakai stopword list yang sama
#         token_pattern=r"(?u)\b[^\d\W]+\b",
#         min_df=1,                     # <- turunkan drastis, sub-cluster jumlahnya sedikit
#     )

#     sub_umap = UMAP(n_neighbors=10, n_components=5, metric="cosine", min_dist=0.0, random_state=42)
#     sub_hdbscan = HDBSCAN(
#         min_cluster_size=max(10, int(len(idx) * 0.3)),
#         min_samples=5, metric="euclidean",
#         cluster_selection_method="eom", prediction_data=True,
#     )

#     try:
#         sub_model = BERTopic(
#             embedding_model=None, umap_model=sub_umap, hdbscan_model=sub_hdbscan,
#             vectorizer_model=sub_vectorizer, calculate_probabilities=False, verbose=False,
#         )
#         sub_topics, _ = sub_model.fit_transform(sub_documents, sub_embeddings)
#         hidden_outlier_mask = np.array(sub_topics) == -1
#         refined_topics[idx[hidden_outlier_mask]] = -1
#     except Exception as e:
#         skipped_topics.append((t, str(e)))
#         continue

# print(f"Outlier awal (tahap 1)              : {(topics_array == -1).sum():,} ({(topics_array==-1).sum()/len(topics_array):.2%})")
# print(f"Outlier setelah refinement (tahap 2): {(refined_topics == -1).sum():,} ({(refined_topics==-1).sum()/len(refined_topics):.2%})")
# print(f"Topik yang di-skip (gagal sub-cluster): {len(skipped_topics)}")
# if skipped_topics:
#     print(skipped_topics[:5])

In [400]:
# for fraction in [0.10, 0.15, 0.20, 0.30]:
#     refined_topics = topics_array.copy()
    
#     for t in set(topics_array):
#         if t == -1:
#             continue
#         idx = np.where(topics_array == t)[0]
#         if len(idx) < 20:
#             continue
        
#         sub_embeddings = embeddings[idx]
#         sub_documents = [documents[i] for i in idx]
        
#         sub_vectorizer = CountVectorizer(
#             ngram_range=(1, 2), stop_words=pure_stopwords,
#             token_pattern=r"(?u)\b[^\d\W]+\b", min_df=1,
#         )
#         sub_umap = UMAP(n_neighbors=10, n_components=5, metric="cosine", min_dist=0.0, random_state=42)
#         sub_hdbscan = HDBSCAN(
#             min_cluster_size=max(10, int(len(idx) * fraction)),
#             min_samples=5, metric="euclidean",
#             cluster_selection_method="eom", prediction_data=True,
#         )
        
#         try:
#             sub_model = BERTopic(
#                 embedding_model=None, umap_model=sub_umap, hdbscan_model=sub_hdbscan,
#                 vectorizer_model=sub_vectorizer, calculate_probabilities=False, verbose=False,
#             )
#             sub_topics, _ = sub_model.fit_transform(sub_documents, sub_embeddings)
#             hidden_outlier_mask = np.array(sub_topics) == -1
#             refined_topics[idx[hidden_outlier_mask]] = -1
#         except Exception:
#             continue
    
#     pct = (refined_topics == -1).sum() / len(refined_topics) * 100
#     print(f"fraction={fraction} -> outlier setelah refinement: {pct:.2f}%")